
# CatBoost 训练与预测（5 折 CV + 防泄漏）
> 文件名：`catboosting.ipynb`  
> 流程：读取主表 → 统一时间为“天数” → 合并流水聚合特征 → 指定更多 `cat_features` → 5折分层CV训练 → OOF/重要性/测试预测落盘。

**要点**
- 时间字段（`issue_time/record_time/history_time`）统一转换为**距离基准日**的天数（`2025-08-31`），并构造**简单时间差分**；
- `level` 额外拆出 `grade/subgrade`，并把 `title/career/zip_code/residence/term/syndicated/installment/level/grade/subgrade` 全部作为**类别特征**；
- 合并流水特征时**不带 label**，对缺失流水的 id 用 0 填充并加 `stm_missing`；
- `StratifiedKFold` + `scale_pos_weight` + 早停，追求稳健泛化；
- 提供 `DROP_FEATURES` 开关，方便你快速“去掉一些特征”验证；
- 产物：`outputs/oof_predictions.csv`、`outputs/feature_importance.csv`、`outputs/test_pred_catboost.csv`。


In [1]:

# 可选：如本机未安装 catboost，可先执行（建议在你自己的环境运行，不在此环境执行）：
# !pip install -U catboost pandas numpy scikit-learn


In [2]:

# ============ 可配置区 ============
REF_DATE_STR = "2025-08-31"
TRAIN_CSV = "train.csv"
TEST_CSV  = "testab.csv"  # 若没有测试集，可留空文件名或确保文件不存在

TRAIN_STM_FEAT = "train_statement_feature.csv"
TEST_STM_FEAT  = "testab_statement_feature.csv"

OUT_DIR = "outputs"

# 手动屏蔽部分“不稳定/可能有毒”的特征（列名完全匹配）。可按需增删。
DROP_FEATURES = [
    # 例如：'record_time_days', 'diff_issue_record_days'
]

N_FOLDS = 5
RANDOM_STATE = 42
# ==================================


In [3]:

import os, gc, math, random, json
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

try:
    from catboost import CatBoostClassifier, Pool
except Exception as e:
    raise RuntimeError("未找到 catboost，请先在本机安装：pip install catboost") from e

REF_DATE = pd.Timestamp(REF_DATE_STR)

def _to_days_since_now(unix_series):
    dt = pd.to_datetime(unix_series, unit='s', utc=True, errors='coerce').dt.tz_convert(None)
    return (REF_DATE - dt).dt.days

def prepare_main_table(df):
    use_cols = [
        'id','title','career','zip_code','residence','loan','term','interest_rate',
        'issue_time','syndicated','installment','record_time','history_time',
        'total_accounts','balance_accounts','balance_limit','balance','level'
    ] + (['label'] if 'label' in df.columns else [])
    df = df[use_cols].copy()

    # 时间转“距离基准日的天数”
    for c in ['issue_time','record_time','history_time']:
        df[f'{c}_days'] = _to_days_since_now(df[c])

    # 简单差分
    df['diff_issue_record_days']   = df['issue_time_days'] - df['record_time_days']
    df['diff_issue_history_days']  = df['issue_time_days'] - df['history_time_days']
    df['diff_record_history_days'] = df['record_time_days'] - df['history_time_days']

    # 信用使用率与账户占比
    df['utilization'] = (df['balance'] / df['balance_limit']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 10)
    df['accounts_ratio'] = (df['balance_accounts'] / df['total_accounts']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)

    # level 拆 grade/subgrade，并把多列转为字符串类别
    def split_level(x):
        if isinstance(x, str) and len(x) >= 2:
            return x[0], x[1:]
        return 'NA', 'NA'
    lv = df['level'].fillna('NA')
    df['grade'], df['subgrade'] = zip(*lv.map(split_level))

    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    for c in cat_cols:
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = df[c].astype('Int64').astype(str)
        else:
            df[c] = df[c].astype(str)
        df[c] = df[c].fillna('NA')
    return df

def merge_statement_feats(main_df, stm_path):
    if os.path.exists(stm_path):
        stm = pd.read_csv(stm_path)
        # 去除任何 label 列，避免泄漏
        stm = stm[[c for c in stm.columns if c != 'label']].copy()
        main_df = main_df.merge(stm, on='id', how='left')
    else:
        main_df['stm_missing'] = 1

    # 缺失补零（外连接导致）
    num_exclude = ['id','label','title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    stm_num_cols = [c for c in main_df.columns if c not in num_exclude]
    main_df[stm_num_cols] = main_df[stm_num_cols].fillna(0)
    return main_df

def get_feature_lists(df):
    drop_cols = ['id','label']
    features = [c for c in df.columns if c not in drop_cols]
    # 屏蔽不想用的特征
    features = [c for c in features if c not in DROP_FEATURES]
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    cat_cols = [c for c in cat_cols if c in features]
    return features, cat_cols

def train_cv(df_train, features, cat_cols, n_folds=5, seed=1337):
    X = df_train[features]
    y = df_train['label']
    folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    pos = y.sum()
    neg = len(y) - pos
    scale_pos = float(neg / max(pos, 1))

    oof_pred = np.zeros(len(y), dtype=float)
    models = []
    feat_importances = pd.DataFrame(0.0, index=features, columns=['importance'])
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(folds.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        train_pool = Pool(X_tr, label=y_tr, cat_features=[X.columns.get_loc(c) for c in cat_cols])
        valid_pool = Pool(X_va, label=y_va, cat_features=[X.columns.get_loc(c) for c in cat_cols])

        model = CatBoostClassifier(
            iterations=5000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=6,
            loss_function='Logloss',
            eval_metric='AUC',
            bootstrap_type='Bernoulli',  # 改为 'Bernoulli'
            random_seed=seed + fold,
            subsample=0.8,
            rsm=0.8,
            scale_pos_weight=scale_pos,
            early_stopping_rounds=300,
            verbose=200
        )
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=200)

        va_pred = model.predict_proba(valid_pool)[:,1]
        oof_pred[va_idx] = va_pred
        auc = roc_auc_score(y_va, va_pred)
        fold_scores.append(auc)

        fi = pd.Series(model.get_feature_importance(train_pool, type='FeatureImportance'), index=features)
        feat_importances['importance'] += fi

        models.append(model)
        print(f"[Fold {fold}] AUC = {auc:.5f} | Best iters = {model.tree_count_}")

    mean_auc = roc_auc_score(y, oof_pred)
    print(f"[OOF] Mean AUC = {mean_auc:.5f} | Folds = {fold_scores}")

    feat_importances['importance'] /= n_folds
    feat_importances = feat_importances.sort_values('importance', ascending=False)
    return models, oof_pred, feat_importances, fold_scores

def predict_average(models, df, features, cat_cols):
    X = df[features]
    pool = Pool(X, cat_features=[X.columns.get_loc(c) for c in cat_cols])
    preds = None
    for m in models:
        p = m.predict_proba(pool)[:,1]
        preds = p if preds is None else (preds + p)
    preds = preds / max(len(models), 1)
    return preds


In [4]:

# 读取与准备数据
os.makedirs(OUT_DIR, exist_ok=True)

tr = pd.read_csv(TRAIN_CSV)
tr = prepare_main_table(tr)
tr = merge_statement_feats(tr, TRAIN_STM_FEAT)

features, cat_cols = get_feature_lists(tr)
print("训练样本维度：", tr.shape)
print("特征数：", len(features))
print("类别特征：", cat_cols)


训练样本维度： (53480, 50)
特征数： 48
类别特征： ['title', 'career', 'zip_code', 'residence', 'term', 'syndicated', 'installment', 'level', 'grade', 'subgrade']


In [5]:

# 训练与OOF评估
models, oof_pred, feat_importances, fold_scores = train_cv(tr, features, cat_cols, n_folds=N_FOLDS, seed=RANDOM_STATE)

oof_df = pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_pred': oof_pred})
oof_path = os.path.join(OUT_DIR, 'oof_predictions.csv')
fi_path  = os.path.join(OUT_DIR, 'feature_importance.csv')
oof_df.to_csv(oof_path, index=False, encoding='utf-8')
feat_importances.to_csv(fi_path, index=True, encoding='utf-8')
print(f"[SAVE] OOF -> {oof_path} | FI -> {fi_path}")
feat_importances.head(20)


0:	test: 0.6256575	best: 0.6256575 (0)	total: 230ms	remaining: 19m 8s
200:	test: 0.6519378	best: 0.6520080 (199)	total: 10.9s	remaining: 4m 20s
400:	test: 0.6557702	best: 0.6561960 (391)	total: 21.7s	remaining: 4m 8s
600:	test: 0.6595957	best: 0.6596853 (588)	total: 33.4s	remaining: 4m 4s
800:	test: 0.6600707	best: 0.6607315 (694)	total: 44.5s	remaining: 3m 53s
1000:	test: 0.6609846	best: 0.6611543 (986)	total: 55.9s	remaining: 3m 43s
1200:	test: 0.6601029	best: 0.6611543 (986)	total: 1m 7s	remaining: 3m 32s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.6611542545
bestIteration = 986

Shrink model to first 987 iterations.
[Fold 1] AUC = 0.66115 | Best iters = 987
0:	test: 0.5979255	best: 0.5979255 (0)	total: 56.8ms	remaining: 4m 43s
200:	test: 0.6545748	best: 0.6546569 (198)	total: 11.2s	remaining: 4m 26s
400:	test: 0.6575309	best: 0.6579460 (378)	total: 22.7s	remaining: 4m 20s
600:	test: 0.6583597	best: 0.6589828 (482)	total: 34.4s	remaining: 4m 11s
800:	test: 0

,importance
term,5.502084
balance_accounts,5.141325
level,4.956825
interest_rate,4.909463
grade,4.165242
balance,3.595783
issue_time,3.213625
utilization,3.162103
accounts_ratio,3.022656
issue_time_days,2.968441


In [7]:

# （可选）生成测试集预测
if os.path.exists(TEST_CSV):
    te = pd.read_csv(TEST_CSV)
    te = prepare_main_table(te)
    te = merge_statement_feats(te, TEST_STM_FEAT)
    if 'label' in te.columns:
        te = te.drop(columns=['label'])

    preds = predict_average(models, te, features, cat_cols)
    sub = pd.DataFrame({'id': te['id'], 'label': preds})
    sub_path = os.path.join(OUT_DIR, 'test_pred_catboost.csv')
    sub.to_csv(sub_path, index=False, encoding='utf-8')
    print(f"[SAVE] Test predictions -> {sub_path}")
else:
    print(f"[INFO] 未找到 {TEST_CSV}，跳过测试预测。")


[SAVE] Test predictions -> outputs\test_pred_catboost.csv



## 如何做“删特征”实验
- 打开本 Notebook 顶部“可配置区”，往 `DROP_FEATURES` 列表里加入你想屏蔽的列名，再重新执行训练单元；  
- 也可只保留核心特征（额度、使用率、账户占比、若干天数差、流水总量级/频次/比率），由少到多逐步加回；  
- 通过 `outputs/oof_predictions.csv` 的 OOF 结果与你的线上 AUC 对照，选择最稳的一版提交。
